In [ ]:
"""
获取节点数据  从tushare获取特征数据
"""

import os
import pandas as pd
import tushare as ts
from datetime import datetime, timedelta

def get_node_data(input_file, output_file):
    """
    处理股票数据并生成带标签的节点数据
    
    参数:
    input_file (str): 输入CSV文件路径
    output_file (str): 输出CSV文件路径
    """
    # 设置Tushare token (需提前注册获取)
    ts.set_token('e5991012344cb5807859d974da3d1a08a98f5c404bf8dace4e7e4ebe')  # 替换为你的实际token
    pro = ts.pro_api()

    # 1. 读取数据并处理股票代码
    df = pd.read_csv(input_file)
    
    # 合并两列股票代码并去重
    stocks = pd.concat([df['stock_1'], df['stock_2']]).unique().tolist()
    
    # 2. 获取日期列并转换格式
    df['trade_date'] = pd.to_datetime(df['new_date_column']).dt.strftime('%Y%m%d')
    unique_dates = df['trade_date'].unique().tolist()

    ts_code=','.join(stocks)
    # 3. 获取股票数据
    all_data = []
    
    # 获取交易日历（确保包含所有日期）
    cal_df = pro.trade_cal(exchange='', 
                          start_date=min(unique_dates), 
                          end_date=(pd.to_datetime(max(unique_dates)) + timedelta(days=30)).strftime('%Y%m%d'))
    cal_df = cal_df[cal_df['is_open'] == 1].sort_values('cal_date')
    
    # 创建交易日历映射（当日 -> 下一交易日）
    cal_df['next_trade_date'] = cal_df['cal_date'].shift(-1)
    date_map = cal_df.set_index('cal_date')['next_trade_date'].to_dict()
    
    # 获取基础行情数据（当日数据）
    daily_df = pro.daily(ts_code=ts_code, 
                        start_date=min(unique_dates), 
                        end_date=max(unique_dates))
    
    # 获取下一交易日数据
    next_dates = [date_map.get(d, d) for d in unique_dates]
    next_daily_df = pro.daily(ts_code=ts_code, 
                             start_date=min(next_dates), 
                             end_date=max(next_dates))
    next_daily_df = next_daily_df[['ts_code', 'trade_date', 'close']].rename(
        columns={'trade_date': 'next_trade_date', 'close': 'next_close'})
    
    # 合并当日数据和下一交易日数据
    merged_df = pd.merge(daily_df, next_daily_df,
                        left_on=['ts_code', 'trade_date'],
                        right_on=['ts_code', 'next_trade_date'],
                        how='left')
    
    all_data.append(merged_df)
    
    if not all_data:
        raise ValueError("未获取到任何股票数据")
    
    full_data = pd.concat(all_data)
    
    # 5. 计算标签（使用下一交易日收盘价）
    full_data['label'] = (full_data['next_close'] > full_data['close']).astype(int)
    
    # 6. 筛选所需列并保存
    result = full_data[['ts_code','trade_date','open','high','low','close','pre_close','change','pct_chg','vol','amount','label']]
    result.to_csv(output_file, index=False)
    print(f"处理完成，结果已保存至: {output_file}")

# 使用示例
input_dir="../data_processed/all/SS_edge"
output_dir="../data_processed/all/nodes"

for filename in os.listdir(input_dir):
    # 构建输入输出路径
    input_path = os.path.join(input_dir, filename)
    output_filename = filename
    output_path = os.path.join(output_dir, output_filename)
    
    print(f"处理中: {filename} → {output_filename}")
    
    try:
        get_node_data(input_path, output_path)
        
    except Exception as e:
        print(f"处理文件{filename}时出错: {str(e)}")

In [ ]:
import os
import pandas as pd
import tushare as ts
from datetime import datetime, timedelta

In [17]:
# 设置Tushare token (需提前注册获取)
ts.set_token('e5991012344cb5807859d974da3d1a08a98f5c404bf8dace4e7e4ebe')  # 替换为你的实际token
pro = ts.pro_api()

# 1. 读取数据并处理股票代码
df = pd.read_csv("../data_processed/all/SS_edge/news_20220328.csv")

# 合并两列股票代码并去重
stocks = pd.concat([df['stock_1'], df['stock_2']]).unique().tolist()

# 2. 获取日期列并转换格式
df['trade_date'] = pd.to_datetime(df['new_date_column']).dt.strftime('%Y%m%d')
unique_dates = df['trade_date'].unique().tolist()

ts_code=','.join(stocks)
# 3. 获取股票数据
all_data = []

# 获取交易日历（确保包含所有日期）
cal_df = pro.trade_cal(exchange='', 
                        start_date=min(unique_dates), 
                        end_date=(pd.to_datetime(max(unique_dates)) + timedelta(days=30)).strftime('%Y%m%d'))
cal_df = cal_df[cal_df['is_open'] == 1].sort_values('cal_date')

# 创建交易日历映射（当日 -> 下一交易日）
cal_df['next_trade_date'] = cal_df['cal_date'].shift(-1)
date_map = cal_df.set_index('cal_date')['next_trade_date'].to_dict()

# 获取基础行情数据（当日数据）
daily_df = pro.daily(ts_code=ts_code, 
                    start_date=min(unique_dates), 
                    end_date=max(unique_dates))

# 获取下一交易日数据
next_dates = [date_map.get(d, d) for d in unique_dates]
next_daily_df = pro.daily(ts_code=ts_code, 
                            start_date=min(next_dates), 
                            end_date=max(next_dates))
next_daily_df = next_daily_df[['ts_code', 'trade_date', 'close']].rename(
    columns={'trade_date': 'next_trade_date', 'close': 'next_close'})

# 合并当日数据和下一交易日数据
# merged_df = pd.merge(daily_df, next_daily_df,
#                     left_on=['ts_code', 'trade_date'],
#                     right_on=['ts_code', 'next_trade_date'],
#                     how='left')


merged_df = pd.merge(daily_df, next_daily_df)

all_data.append(merged_df)

In [19]:
daily_df

,ts_code,trade_date,open,high,low,close,pre_close,change,pct_chg,vol,amount
0,000001.SZ,20220328,14.80,15.05,14.63,14.85,14.98,-0.13,-0.8678,727136.99,1078327.302
1,000333.SZ,20220328,55.87,56.16,54.80,55.80,56.09,-0.29,-0.5170,215852.82,1198968.556
2,000671.SZ,20220328,3.90,4.25,3.78,4.25,3.86,0.39,10.1036,7454951.79,3054806.383
3,000686.SZ,20220328,7.54,7.64,7.46,7.60,7.57,0.03,0.3963,121959.65,92155.676
4,000713.SZ,20220328,9.80,10.18,9.47,9.67,9.49,0.18,1.8967,772503.56,749997.743
...,...,...,...,...,...,...,...,...,...,...,...
139,688609.SH,20220328,9.33,10.90,9.29,10.90,9.08,1.82,20.0441,350106.65,368911.363
140,688662.SH,20220328,33.60,33.62,32.00,32.06,33.60,-1.54,-4.5833,5261.28,17068.602
141,688687.SH,20220328,26.10,26.65,21.12,21.85,26.40,-4.55,-17.2348,152192.05,348102.943
142,688798.SH,20220328,172.22,174.64,167.11,167.12,174.86,-7.74,-4.4264,1643.56,27829.616


In [20]:
"""
获取节点数据  从tushare获取特征数据
"""

import os
import pandas as pd
import tushare as ts
from datetime import datetime, timedelta

def get_node_data(input_file, output_file):
    """
    处理股票数据并生成带标签的节点数据
    
    参数:
    input_file (str): 输入CSV文件路径
    output_file (str): 输出CSV文件路径
    """
    # 设置Tushare token (需提前注册获取)
    ts.set_token('e5991012344cb5807859d974da3d1a08a98f5c404bf8dace4e7e4ebe')  # 替换为你的实际token
    pro = ts.pro_api()

    # 1. 读取数据并处理股票代码
    df = pd.read_csv(input_file)
    
    # 合并两列股票代码并去重
    stocks = pd.concat([df['stock_1'], df['stock_2']]).unique().tolist()
    
    # 2. 获取日期列并转换格式
    df['trade_date'] = pd.to_datetime(df['new_date_column']).dt.strftime('%Y%m%d')
    unique_dates = df['trade_date'].unique().tolist()

    ts_code=','.join(stocks)
    # 3. 获取股票数据
    all_data = []
    
    # 获取交易日历（确保包含所有日期）
    cal_df = pro.trade_cal(exchange='', 
                          start_date=min(unique_dates), 
                          end_date=(pd.to_datetime(max(unique_dates)) + timedelta(days=30)).strftime('%Y%m%d'))
    cal_df = cal_df[cal_df['is_open'] == 1].sort_values('cal_date')
    
    # 创建交易日历映射（当日 -> 下一交易日）
    cal_df['next_trade_date'] = cal_df['cal_date'].shift(-1)
    date_map = cal_df.set_index('cal_date')['next_trade_date'].to_dict()
    
    # 获取基础行情数据（当日数据）
    daily_df = pro.daily(ts_code=ts_code, 
                        start_date=min(unique_dates), 
                        end_date=max(unique_dates))
    
    # 获取下一交易日数据
    next_dates = [date_map.get(d, d) for d in unique_dates]
    next_daily_df = pro.daily(ts_code=ts_code, 
                             start_date=min(next_dates), 
                             end_date=max(next_dates))
    next_daily_df = next_daily_df[['ts_code', 'trade_date', 'close']].rename(
        columns={'trade_date': 'next_trade_date', 'close': 'next_close'})
    
    # 合并当日数据和下一交易日数据
    merged_df = pd.merge(daily_df, next_daily_df)
    
    all_data.append(merged_df)
    
    if not all_data:
        raise ValueError("未获取到任何股票数据")
    
    full_data = pd.concat(all_data)
    
    # 5. 计算标签（使用下一交易日收盘价）
    full_data['label'] = (full_data['next_close'] > full_data['close']).astype(int)
    
    # 6. 筛选所需列并保存
    result = full_data[['ts_code','trade_date','open','high','low','close','pre_close','change','pct_chg','vol','amount','label']]
    result.to_csv(output_file, index=False)
    print(f"处理完成，结果已保存至: {output_file}")

# 使用示例
input_dir="../data_processed/all/SS_edge"
output_dir="../data_processed/all/nodes"

for filename in os.listdir(input_dir):
    # 构建输入输出路径
    input_path = os.path.join(input_dir, filename)
    output_filename = filename
    output_path = os.path.join(output_dir, output_filename)
    
    print(f"处理中: {filename} → {output_filename}")
    
    try:
        get_node_data(input_path, output_path)
        
    except Exception as e:
        print(f"处理文件{filename}时出错: {str(e)}")

处理中: news_20220328.csv → news_20220328.csv
处理完成，结果已保存至: ../data_processed/all/nodes\news_20220328.csv
处理中: news_20220329.csv → news_20220329.csv
处理完成，结果已保存至: ../data_processed/all/nodes\news_20220329.csv
处理中: news_20220330.csv → news_20220330.csv
处理完成，结果已保存至: ../data_processed/all/nodes\news_20220330.csv
处理中: news_20220331.csv → news_20220331.csv
处理完成，结果已保存至: ../data_processed/all/nodes\news_20220331.csv
处理中: news_20220402.csv → news_20220402.csv
处理完成，结果已保存至: ../data_processed/all/nodes\news_20220402.csv
处理中: news_20220404.csv → news_20220404.csv
处理完成，结果已保存至: ../data_processed/all/nodes\news_20220404.csv
处理中: news_20220405.csv → news_20220405.csv
处理完成，结果已保存至: ../data_processed/all/nodes\news_20220405.csv
处理中: news_20220406.csv → news_20220406.csv
处理完成，结果已保存至: ../data_processed/all/nodes\news_20220406.csv
处理中: news_20220407.csv → news_20220407.csv
处理完成，结果已保存至: ../data_processed/all/nodes\news_20220407.csv
处理中: news_20220408.csv → news_20220408.csv
处理完成，结果已保存至: ../data_processed/all/node

In [ ]:
import torch
import os
import pandas as pd
from torch_geometric.data import HeteroData
from datetime import datetime
import numpy as np
from torch_geometric.loader import DataLoader
import torch.nn.functional as F
from torch_geometric.nn import HeteroConv, SAGEConv, Linear,GATv2Conv,GCNConv
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'  # 启用详细错误报告
os.environ['TORCH_USE_CUDA_DSA'] = '1'   # 启用设备端断言
from HeteroGraphSL import build_or_load_dataset
# 构建或加载数据集
train_graphs, test_graphs = build_or_load_dataset()

# 4. 创建批次数据加载器
batch_size = 16  # 可以根据GPU内存调整

train_loader = DataLoader(train_graphs, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_graphs, batch_size=batch_size, shuffle=False)

import torch
import os
from torch_geometric.data import HeteroData
from torch_geometric.loader import DataLoader
import torch.nn.functional as F
from torch_geometric.nn import HeteroConv, SAGEConv, Linear
from sklearn.metrics import accuracy_score
from HeteroGraphSL import build_or_load_dataset
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'  # 启用详细错误报告
os.environ['TORCH_USE_CUDA_DSA'] = '1'   # 启用设备端断言


# 构建或加载数据集
train_graphs, test_graphs = build_or_load_dataset()

# 创建批次数据加载器
batch_size = 2
train_loader = DataLoader(train_graphs, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_graphs, batch_size=batch_size, shuffle=False)

class SimpleHeteroGNN(torch.nn.Module):
    def __init__(self, hidden_channels, out_channels):
        super().__init__()
        # 初始化线性变换
        self.lin_stock = Linear(8, hidden_channels)
        self.lin_news = Linear(1, hidden_channels)
        self.lin_industry = Linear(1, hidden_channels)
        # 初始化HeteroConv层
        conv_dict = {
            ('stock', 'link', 'stock'): SAGEConv((-1, -1), hidden_channels),
            ('news', 'sentiment', 'stock'): SAGEConv((-1, -1), hidden_channels),
            ('industry', 'marketratio', 'stock'): SAGEConv((-1, -1), hidden_channels),
        }
        self.conv = HeteroConv(conv_dict, aggr='mean')
        # 最终分类器
        self.classifier = Linear(hidden_channels, out_channels)

    def forward(self, x_dict, edge_index_dict):
        # 特征变换
        x_dict = {
            'stock': F.relu(self.lin_stock(x_dict['stock'])),
            'news': F.relu(self.lin_news(x_dict['news'])),
            'industry': F.relu(self.lin_industry(x_dict['industry']))
        }
        # 应用卷积层
        x_dict = self.conv(x_dict, edge_index_dict)
        for key in x_dict:
            x_dict[key] = F.relu(x_dict[key])
        # 只返回股票节点预测
        return self.classifier(x_dict['stock'])

def train(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    total_samples = 0
    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        out = model(batch.x_dict, batch.edge_index_dict)
        y = batch['stock'].y
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * y.size(0)
        total_samples += y.size(0)
    return total_loss / total_samples

def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    total_samples = 0
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            out = model(batch.x_dict, batch.edge_index_dict)
            y = batch['stock'].y
            loss = criterion(out, y)
            total_loss += loss.item() * y.size(0)
            total_samples += y.size(0)
            preds = out.argmax(dim=1).cpu().numpy()
            labels = y.cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels)
    acc = accuracy_score(all_labels, all_preds)
    avg_loss = total_loss / total_samples
    return avg_loss, acc

def main():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"使用设备: {device}")
    # 初始化模型
    model = SimpleHeteroGNN(
        hidden_channels=16,
        out_channels=2
    ).to(device)
    # 打印模型结构
    print("模型结构:")
    print(model)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    criterion = torch.nn.CrossEntropyLoss()
    # 完整训练
    print("\n开始训练...")
    for epoch in range(1, 500):
        train_loss = train(model, train_loader, optimizer, criterion, device)
        test_loss, test_acc = evaluate(model, test_loader, criterion, device)
        print(f'Epoch: {epoch:03d}, Train Loss: {train_loss:.4f}, '
              f'Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.4f}')

if __name__ == "__main__":
    main()